# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 dataset using the `mlcroissant` library. 

### Dataset Source
The dataset is described by a Croissant schema and is accessible via the following URL:
```
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load the dataset's metadata and prepare for record exploration using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata fields
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}\n")
print(f"Identifier: {metadata.identifier}")
print(f"Keywords: {getattr(metadata, 'keywords', None)}")
print(f"Spatial Coverage: {getattr(metadata, 'spatialCoverage', None)}")
print(f"Temporal Coverage: {getattr(metadata, 'temporalCoverage', None)}")

## 2. Data Overview

Let us review the available record sets and their structure. We'll list all record sets (tables/files), each one's `@id`, and then fetch their fields and columns, all by `@id` as per Croissant best practices.

In [ ]:
# List all record sets available in the dataset

record_sets = list(dataset.record_sets)

if not record_sets:
    print("No record sets found in the Croissant schema.")
else:
    print("Record sets present in the dataset:")
    for rs in record_sets:
        print(f"- Record set name: {getattr(rs, 'name', None)}, @id: {getattr(rs, '@id', None)}")
        # List field @id's if fields exist
        if hasattr(rs, 'fields'):
            print("  Fields (by @id):")
            for field in rs.fields:
                print(f"    - {getattr(field, '@id', None)}")
        # List column @id's if columns exist
        if hasattr(rs, 'columns'):
            print("  Columns (by @id):")
            for column in rs.columns:
                print(f"    - {getattr(column, '@id', None)}")

## 3. Data Extraction

We'll extract data for each available record set using their `@id`. Each dataset is loaded into a pandas DataFrame for convenience, and we inspect the column names for each.

In [ ]:
# If record sets are found, extract them by @id into a dictionary of DataFrames
dataframes = {}

if not record_sets:
    print("No record sets to extract.")
else:
    # Get the @id for each record set
    record_set_ids = [getattr(rs, '@id', None) for rs in record_sets]

    for rs_id in record_set_ids:
        # Load all records for the record set by @id
        try:
            records = list(dataset.records(record_set=rs_id))
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded DataFrame for record set @id: {rs_id}")
            print(df.columns.tolist())
            print(df.head(), "\n")
        except Exception as e:
            print(f"Could not load record set {rs_id}: {e}")
    # Choose first record set as example for further analysis
    example_rs_id = record_set_ids[0] if record_set_ids else None

## 4. Exploratory Data Analysis (EDA)

Let us perform exploratory data analysis:
- Filter records based on a numeric field,
- Normalize values for that field,
- Group by a categorical field if available.

Remember to use the field/column `@id`s for all references.

In [ ]:
# For demonstration, choose one record set
# Please update 'example_rs_id' to your desired record set @id if known
if 'example_rs_id' not in locals() or example_rs_id is None:
    print("No example record set ID found. Skipping EDA.")
else:
    df = dataframes[example_rs_id]
    print(f"Available columns (@id) in DataFrame for record set '{example_rs_id}':")
    print(df.columns.tolist())
    
    # Attempt to pick a numeric field by pandas 'number' dtype or fallback
    numeric_field = None
    for col in df.columns:
        try:
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field = col
                break
        except Exception:
            continue
    if numeric_field is None and len(df.columns) > 0:
        numeric_field = df.columns[0]  # fallback

    print(f"\nUsing field '@id': {numeric_field} for numeric EDA.")
    # Drop NA for this demo
    filter_col = numeric_field
    df = df.dropna(subset=[filter_col])

    try:
        # Convert if not already numeric
        df[filter_col] = pd.to_numeric(df[filter_col], errors='coerce')
        df = df.dropna(subset=[filter_col])
        threshold = df[filter_col].quantile(0.75)  # Use 75th percentile as example threshold
        filtered_df = df[df[filter_col] > threshold]
        print(f"Filtered records with {filter_col} > {threshold}:")
        print(filtered_df.head())

        normalized_col = f"{filter_col}_normalized"
        filtered_df[normalized_col] = (filtered_df[filter_col] - filtered_df[filter_col].mean()) / filtered_df[filter_col].std()
        print(f"\nNormalized {filter_col} for filtered records:")
        print(filtered_df[[filter_col, normalized_col]].head())
    except Exception as e:
        print(f"Could not perform numeric EDA: {e}")

    # Try to find a grouping field (categorical)
    group_field = None
    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]) and col != numeric_field:
            group_field = col
            break
    if group_field:
        try:
            # group by and compute mean (for numeric columns)
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"\nGrouped data by {group_field} (only numeric columns shown):")
            print(grouped_df.head())
        except Exception as e:
            print(f"Grouping failed: {e}")
    else:
        print("No categorical group field found for grouping.")

## 5. Visualization

Let's visualize the distribution of the selected numeric field and any grouping variable if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'filtered_df' in locals() and numeric_field is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field], kde=True, bins=20)
    plt.title(f"Distribution of '{numeric_field}' in Filtered Data")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
    
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=filtered_df, x=group_field, y=numeric_field)
        plt.title(f"'{numeric_field}' by '{group_field}'")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("No EDA results to visualize.")

## 6. Conclusion

Using `mlcroissant`, we've loaded metadata and records from the FAIR² dataset described by a Croissant schema. We have:
- Explored available record sets and fields by their `@id`,
- Loaded and previewed data in pandas DataFrames,
- Performed simple filtering, normalization, and grouping as a demonstration of EDA,
- Visualized the distribution of a numeric field and explored groupwise statistics.

To go further, refine field/column selections by consulting the data dictionary (in the Croissant schema), and tailor filtering/grouping to analytical questions relevant to rangeland management practices.